# 3단계 실험: RAG → Claude 초안 (Writing Chain)

Chroma 에 저장된 자료를 retrieve 해서 Claude 로 한국어 블로그 초안을 생성합니다.

**전제**: 2단계 노트북에서 `index_research()` 를 한 번 이상 실행해 `data/chroma/` 컬렉션에 자료가 들어 있어야 합니다. (아래 1번 셀에서 보강 가능)

## 0. 환경 설정

In [1]:
import os, sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from dotenv import load_dotenv
load_dotenv(PROJECT_ROOT / ".env")

for k in ("OPENAI_API_KEY", "ANTHROPIC_API_KEY", "TAVILY_API_KEY"):
    print(f"{k}: {'OK' if os.getenv(k) else 'MISSING'}")

OPENAI_API_KEY: OK
ANTHROPIC_API_KEY: OK
TAVILY_API_KEY: OK


## 1. 컬렉션 상태 확인 (필요시 인덱싱 보강)

비어 있으면 즉석에서 1+2 단계를 돌려 자료를 채웁니다.

In [2]:
from llm_jacky.rag import _vector_store, index_research
from llm_jacky.research import run_research

TOPIC = "외국인을 위한 한국 길거리 음식 추천"

vs = _vector_store()
current = len(vs.get().get("ids", []))
print("current chunks in collection:", current)

if current == 0:
    print("-> empty. running research + indexing now...")
    result = run_research(TOPIC, max_results=5)
    n = index_research(result)
    print(f"indexed {n} chunks for topic: {TOPIC}")

current chunks in collection: 16


## 2. Retrieval 미리보기

Claude 에 들어갈 컨텍스트 후보를 먼저 확인합니다.

In [3]:
from llm_jacky.rag import get_retriever
from llm_jacky.writing import _format_context

retriever = get_retriever(k=5)
docs = retriever.invoke(TOPIC)

print(f"retrieved: {len(docs)}\n")
for i, d in enumerate(docs, 1):
    print(f"[{i}] {d.metadata.get('title','')[:60]} | {d.metadata.get('url')}")

print("\n--- formatted context (preview) ---")
print(_format_context(docs)[:800], "...")

retrieved: 5

[1] Top 10 Korean Street Foods You Must Try - Love U Korea | https://loveukorea.com/ko/top-10-korean-street-foods-you-must-try/
[2] 외국인이 좋아하는 한식 49선 - 램프쿡 | https://www.lampcook.com/cook/food_top100_list1.php
[3] [외국인 남편과 한국여행] 외국인이 좋아하는 한국음식 추천! 꼭 먹어봐야 하는 한식들 놓치지 마세요~ : 네 | https://m.blog.naver.com/wjwj38/223176755965
[4] 세계 길거리 음식 BEST 10 — 한국인이 꼭 먹어봐야 할 이색 메뉴 : 국제 : 기독일보 | https://www.christiandaily.co.kr/news/159366
[5] 우리에겐 일상인데 외국인이 보면 충격받는 한국 길거리 풍경[zum 펌글] : 네이버 블로그 | https://m.blog.naver.com/namis8000/222230972605

--- formatted context (preview) ---
[1] Top 10 Korean Street Foods You Must Try - Love U Korea
URL: https://loveukorea.com/ko/top-10-korean-street-foods-you-must-try/
Love U Korea

Top 10 Korean Street Foods You Must Try

# 꼭 먹어봐야 할 한국 길거리 음식 10가지

한국에는 지역, 도시마다 길거리에 다양한 음식이 있습니다. 물론 모든 음식이 맛있겠지만 이중에서 가장 맛있고 한국인에게 인기있는 음식종류와 순위를 보여드리겠습니다. 한국에 방문하게되면 꼭 드셔보시길 추천드립니다. 호불호가 갈릴 수 있으니 하나 시켜서 여러명에서 나눠먹은다음 맛있으면 추가로 주문해도 좋습니다.

## 1위 떡볶이

(Photo source: 약사겸 여행인

## 3. 초안 생성 (write_draft)

기본 모델은 `claude-sonnet-4-6`. 다른 모델로 바꾸려면 `model=` 인자를 넘기세요.

In [5]:
from llm_jacky.writing import write_draft

draft = write_draft(TOPIC, k=5)

print("=== DRAFT ===\n")
print(draft.draft)
print("\n=== SOURCES ===")
for i, s in enumerate(draft.sources, 1):
    print(f"[{i}] {s['title']}\n    {s['url']}")

=== DRAFT ===

# 한국 길거리 음식 완전 정복: 외국인이 꼭 먹어봐야 할 메뉴 추천

## 들어가며

한국을 여행한다면 길거리 음식은 절대 빠뜨릴 수 없는 경험입니다. 한국에는 지역과 도시마다 다양한 길거리 음식이 있으며, 저렴한 가격에 현지 문화를 온몸으로 느낄 수 있는 최고의 방법이기도 합니다.[1] 지금부터 외국인 여행자에게 특히 추천하는 대표 메뉴들을 소개해 드릴게요!

---

## 🌶️ 1. 떡볶이 — 한국 길거리 음식의 왕

한국 길거리 음식을 대표하는 음식으로, 매콤하고 달콤한 소스에 쫀득한 떡과 어묵, 파, 양배추 등이 어우러진 요리입니다.[1] 특히 10대부터 30대 여성들에게 큰 인기를 자랑하며, 외국인이 선호하는 한식 순위에서도 5위에 오를 만큼 세계적으로 인정받고 있습니다.[2] 떡볶이를 주문할 때는 튀김, 순대, 오뎅국과 함께 곁들여 먹으면 더욱 맛있습니다.[1]

---

## 🍙 2. 김밥 — 한국식 롤, 한 끼 식사로도 완벽

일본에 스시가 있다면 한국에는 김밥이 있습니다.[1] 김 안에 밥, 당근, 계란, 햄, 시금치, 단무지 등이 들어 있어 영양도 풍부합니다. 야채김밥, 소고기김밥, 참치김밥, 치즈김밥 등 종류가 다양하므로 주문 전에 어떤 종류가 있는지 확인해 보세요.[1] 외국인 선호 한식 순위에서도 7위를 차지한 검증된 메뉴입니다.[2]

---

## 🔥 3. 삼겹살 & 불고기 — 길거리를 넘어 한국 식문화의 상징

길거리 음식은 아니지만, 한국 여행에서 절대 빠질 수 없는 메뉴입니다. 불고기는 외국인 선호 한식 4위, 삼겹살은 6위에 오를 만큼 외국인들에게 큰 사랑을 받고 있습니다.[2] 실제로 한국을 방문한 외국인들이 "한국 여행 전에는 코리안 바베큐밖에 없다고 생각했는데, 여행 후에는 맛있는 음식이 너무 많다"고 감탄하는 경우가 많습니다.[3]

---

## 🍱 4. 비빔밥 — 외국인이 가장 사랑하는 한식 1위

외국인이 선호하는 한식 1위는 바로 비빔밥입니다.[2] 다양한 채소와 고기, 고추장이 한 

## 4. 프롬프트 변형 비교

`DRAFT_PROMPT` 를 직접 갈아끼워 초안 톤/구조를 실험합니다.

In [6]:
from langchain_anthropic import ChatAnthropic
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

alt_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a travel-blog writer for English-speaking foodies. "
                "Use only the provided context. Cite with [n]."),
    ("human", "Topic: {topic}\n\nContext:\n{context}\n\n"
              "Write a 600-word English blog post in markdown with H1, H2 sections, and a References list."),
])

llm = ChatAnthropic(model="claude-sonnet-4-6", temperature=0, max_tokens=2000)
chain = alt_prompt | llm | StrOutputParser()
print(chain.invoke({"topic": TOPIC, "context": _format_context(docs)}))

# The Ultimate Korean Street Food Guide for Hungry Travelers

If you've ever wandered through the bustling alleyways of Seoul, Busan, or any Korean city, you already know that the street food scene here is nothing short of extraordinary. From spicy rice cakes to seaweed-wrapped rolls, Korean street food is a world unto itself — and it's one that every foodie traveler absolutely *must* explore. Whether you're a first-timer or a seasoned Korea hand, here's your essential guide to the best bites you can grab on the go.

---

## 🌶️ Tteokbokki — The Undisputed King of Korean Street Food

Let's start at the very top. If there is one dish that defines Korean street food culture, it is **tteokbokki** (spicy rice cakes). Ranked as the number one Korean street food, this iconic dish features chewy rice cakes simmered in a gloriously spicy and sweet sauce, often accompanied by fish cakes, green onions, and cabbage [1].

The dish is beloved by Koreans of all ages, but it holds a particularly speci

## 5. 초안 파일로 저장 (선택)

리뷰/공유용. `data/drafts/` 아래에 마크다운으로 떨어집니다.

In [7]:
import re
from datetime import datetime

out_dir = PROJECT_ROOT / "data" / "drafts"
out_dir.mkdir(parents=True, exist_ok=True)
slug = re.sub(r"[^a-zA-Z0-9가-힣]+", "-", TOPIC).strip("-")[:40]
fname = f"{datetime.now().strftime('%Y%m%d-%H%M%S')}-{slug}.md"
path = out_dir / fname
path.write_text(draft.draft, encoding="utf-8")
print("saved:", path)

saved: /Users/jackykim/project/llm-jacky/data/drafts/20260506-192627-외국인을-위한-한국-길거리-음식-추천.md
